### 使用LCEL来自定义路由链

In [2]:
# 导入必要的库
from langchain_core.output_parsers import StrOutputParser  # 导入字符串输出解析器
from langchain_core.prompts import PromptTemplate # 导入提示模板
from langchain_core.runnables import RunnableLambda
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
import os

load_dotenv()

ds_llm = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

# 创建分类链 - 用于确定问题类型
chain = (
    # 创建提示模板，要求模型将问题分类为LangChain、Anthropic或Other
    PromptTemplate.from_template(
        """
            根据下面的用户问题，将其分类为 `LangChain`、`Anthropic`或 `0ther`。

            请只回复一个词作为答案。

            <question>
                {question}
            </question>

            分类结果：
        """
    )
    | ds_llm # 将提示发送给deepseek 模型
    | StrOutputParser()  # 解析模型的输出为纯文本
)

# 创建LangChain专家链 - 模拟Harrison Chase(LangChain创始人)的回答风格
langchain_chain = PromptTemplate.from_template(
    """
    你将扮演一位LangChain专家。请以他的视角回答问题。
    你的回答必须以"正如Harrison Chase告诉我的"开头，否则你会受到惩罚。
    请回答以下问题:

    问题：{question}
    回答："""
) | ds_llm  # 将提示发送给Claude模型 

# 创建Anthropic专家链- 模拟Dario Amodei(Anthropic创始人)的回答风格
anthropic_chain = PromptTemplate.from_template(
    """
        你将扮演一位Anthropic专家。请以他的视角回答问题。
        你的回答必须以"正如Dario Amodei告诉我的"开头，否则你会受到惩罚。
        请回答以下问题：

        问题：{question}
        回答：
    """
) | ds_llm

# 创建通用回答链 - 用于处理其他类型的问题
general_chain = PromptTemplate.from_template(
    """
        请回答以下问题：

        问题：{question}
        回答：
    """
) | ds_llm

# 自定义路由函数 - 根据问题分类结果选择合适的回答链
def route(info):
    print(info)  # 打印分类结果
    # 根据分类结果选择相应的专家链
    if "anthropic" in info["topic"].lower():  # 如果问题与Anthropic相关
        print("claude")
        return anthropic_chain.invoke(info)   # 使用Anthropic专家链
    elif "langchain" in info["topic"].lower(): # 如果问题与LangChain相关
        print("langchain")
        return langchain_chain.invoke(info)  # 使用LangChain专家链
    else:  # 其他类型的问题
        print("general")
        return general_chain.invoke(info)  # 使用通用回答链

# 创建完整的处理链
# 1. 首先将问题分类并保留原始问题
# 2. 然后根据分类结果路由到响应的专家链处理
full_chain = {"topic":chain, "question": lambda x:x["question"]} | RunnableLambda(route)

# 调用完整链处理用户问题
# 这个问题会被分类为Anthropic相关，然后由anthropic_chain处理
full_chain.invoke({"question": "我该如何使用claude？"})


{'topic': 'Anthropic', 'question': '我该如何使用claude？'}
claude


AIMessage(content='正如Dario Amodei告诉我的，使用 Claude 的关键不是把它当成搜索框，而是把它当成一个可以反复协作、推理和打磨的“思考伙伴”。你可以这样上手：\n\n1. **选对入口**\n   - 日常对话、写作、分析：用 Claude.ai 网页版或手机/桌面应用。\n   - 企业集成、批量任务、产品嵌入：用 Anthropic API，或通过 Amazon Bedrock、Google Vertex AI 等云平台调用。\n   - 编程协作：可关注 Claude Code 这类面向开发工作流的使用方式。\n\n2. **把提示写清楚**\n   给 Claude 四类信息：背景、目标、约束、输出格式。比如：\n   - “背景：我在给非技术管理层写一份 AI 安全简报。”\n   - “目标：解释风险并给出三条可执行建议。”\n   - “约束：800 字以内，不用术语，语气稳健。”\n   - “格式：先摘要，再分点，最后给一页式结论。”\n\n3. **提供材料，而不是只问问题**\n   上传文档、表格、代码或会议记录，让 Claude 基于材料工作。长文档场景下，先让它列提纲、标出不确定处，再逐段深入。\n\n4. **用“迭代”而不是“一次成型”**\n   第一轮让 Claude 理解任务并提问；第二轮让它给初稿；第三轮要求它自我审查：事实是否可靠、逻辑是否有漏洞、是否满足约束；最后再让它按你的风格改写。\n\n5. **善用结构化工作流**\n   - 写作：先列大纲 → 再写草稿 → 再做事实/逻辑审查 → 最后润色。\n   - 分析：先定义问题 → 列假设 → 找证据 → 给反方观点 → 输出建议。\n   - 编程：先让 Claude 解释需求与边界 → 给最小实现 → 写测试 → 审查安全与边界条件。\n   - 学习：让它用费曼法解释、出题、批改你的答案，而不是只给结论。\n\n6. **明确要求它承认不确定**\n   可以加一句：“如果信息不足，请先列出你需要我补充的内容；不要编造。” 对事实、数字、引用、法律/医疗/金融建议，务必人工核验。\n\n7. **注意隐私与合规**\n   不要输入不必要的敏感个人信息、密钥、客户数据或未公开机密。企业使用应走相应企业版/API/云平台配置，并

In [3]:
full_chain.invoke({"question": "我该如何使用openai？"})

{'topic': 'Other', 'question': '我该如何使用openai？'}
general


AIMessage(content='使用 OpenAI 主要分两种场景：**普通用户使用 ChatGPT**，和**开发者调用 OpenAI API**。\n\n## 1. 普通用户：使用 ChatGPT\n1. 打开官方网站：`https://chatgpt.com`，或下载官方 ChatGPT 手机应用。\n2. 注册/登录 OpenAI 账户。\n3. 根据需求选择免费版或订阅版。\n4. 在对话框里输入问题即可使用，例如写作、翻译、编程、总结、学习等。\n5. 注意不要输入密码、身份证号、公司机密等敏感信息。\n\n> OpenAI 服务在部分国家和地区可能不可用，请遵守当地法律和 OpenAI 使用条款。\n\n## 2. 开发者：调用 OpenAI API\n适合把 AI 集成到自己的网站、App、机器人或工作流中。\n\n### 基本步骤\n1. 访问 `https://platform.openai.com` 注册开发者账户。\n2. 在后台创建 **API Key**。\n3. 配置计费方式，API 通常按使用量计费，和 ChatGPT 订阅是分开的。\n4. 安装官方 SDK，例如 Python：\n   ```bash\n   pip install openai\n   ```\n5. 设置环境变量，避免把密钥写死在代码里：\n   ```bash\n   export OPENAI_API_KEY="你的API密钥"\n   ```\n6. 调用示例：\n   ```python\n   from openai import OpenAI\n\n   client = OpenAI()\n\n   response = client.chat.completions.create(\n       model="gpt-4o-mini",\n       messages=[\n           {"role": "user", "content": "你好，请介绍一下你自己"}\n       ]\n   )\n\n   print(response.choices[0].message.content)\n   ```\n\n## 3. 使用建议\n- **不要泄露 API Key**，不要提交到 GitHub。\n- 设置用量上

In [4]:
full_chain.invoke({"question": "我该如何使用langchain？"})

{'topic': 'LangChain', 'question': '我该如何使用langchain？'}
langchain


AIMessage(content='正如Harrison Chase告诉我的，使用 LangChain 的关键是：**先把“模型调用”抽象成可组合的组件，再用 Chain/LCEL 把 Prompt、模型、解析器、检索器、工具串起来；从最小可用示例开始，逐步加 RAG、Memory、Agent 和可观测性。**\n\n一个实用入门路径如下：\n\n1. **安装依赖**\n```bash\npip install langchain langchain-openai\n```\n\n2. **配置模型**\n```python\nimport os\nfrom langchain_openai import ChatOpenAI\n\nos.environ["OPENAI_API_KEY"] = "你的 key"\n\nllm = ChatOpenAI(model="gpt-4o-mini", temperature=0)\n```\n\n3. **用 LCEL 构建最小链**\n```python\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom langchain_core.output_parsers import StrOutputParser\n\nprompt = ChatPromptTemplate.from_messages([\n    ("system", "你是一个简洁的中文助手。"),\n    ("human", "{question}")\n])\n\nchain = prompt | llm | StrOutputParser()\n\nanswer = chain.invoke({"question": "用一句话解释 LangChain 是什么"})\nprint(answer)\n```\n\n4. **按需求扩展**\n- **RAG 检索增强**：Document Loader → Text Splitter → Embeddings → VectorStore → Retriever → Prompt → LLM。\n- **对话记忆**：使用消息历史或 LangGraph 管理多轮状态。\n- **工具/Agent**：把函数封装成 Tool，让模型

高级应用

In [20]:
# 导入必要的库
from unittest.mock import patch  # 导入 mock 库，用于模拟函数行为
# from langchain_anthropic import ChatAnthropic  # 导入 Anthropic 的模型接口
from langchain_deepseek import ChatDeepSeek # 导入ds的模型接口
from langchain_openai import ChatOpenAI  # 导入 OpenAI 的语言模型接口
from openai import RateLimitError  # OpenAI的速率限制错误类型
from dotenv import load_dotenv

import httpx # HTTP客户端库
import os

load_dotenv()

# 创建模拟HTTP请求和响应对象，用于构造模拟器的API错误
request = httpx.Request("Get", "/")  # 创建一个GET请求
response = httpx.Response(200, request=request) # 创建一个状态码为200的响应
# 创建一个OpenAI速率限制错误对象，用于模拟API调用超出速率限制的情况
error = RateLimitError("rate limit", response=response, body="")

# 初始化OpenAI模型
# 注意：设置max_retries = 0 是为了避免在遇到速率限制等错误时自动重试
qw_llm = ChatOpenAI(
    model="qwen3.8-max",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
    max_retries=0,
)

# 初始化 deepseek 模型作为备选
ds_llm  = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
    max_retries=0,
)

# 创建带有备用选项的语言模型
# 如果主模型失败，将自动尝试备用模型
llm = qw_llm.with_fallbacks([ds_llm])

# 如果不设置回退机制，当API调用超量就会直接报错
# with patch.object(qw_llm.client, "create", side_effect=error)::
#     try:
#         print(llm.invoke("Why did the chicken cross the road?"))
#     except RateLimitError:
#         print("Hit error")

# 测试备用机制 - 使用中文问题
# 使用patch模拟openai调用失败（抛出速率限制错误）
with patch.object(qw_llm.client, "create", side_effect=error):
    try:
        # 尝试调用语言模型回答中文问题
        # 由于OpenAI被模拟为失败，应该自动切换到备用模型上
        print(llm.invoke("为什么程序员要学会Python?"))
    except RateLimitError:
        # 如果仍然遇到错误（备用机制失败），则打印错误信息
        print("Hit error")

content='程序员不一定要“精通”Python，但很值得学会。核心原因是：Python 性价比高，能用较少时间解决很多实际问题，尤其适合做“第二语言”和工具型语言。\n\n### 1. 上手快，表达力强\nPython 语法简洁、可读性高，写同样功能通常代码更少。程序员可以把精力放在解决问题上，而不是语言细节上。适合快速验证想法、写原型、做脚本。\n\n### 2. 自动化和脚本能力极强\n日常开发中大量重复工作都可以用 Python 自动化：\n- 文件批量处理、日志分析\n- 爬取/整理数据\n- 调用 API、定时任务\n- 构建和部署辅助脚本\n- 测试工具、运维脚本\n\n对任何方向的程序员，Python 都是很实用的“瑞士军刀”。\n\n### 3. 生态庞大，第三方库丰富\nPython 有海量成熟库：\n- 数据处理：pandas、numpy\n- 机器学习/深度学习：scikit-learn、PyTorch、TensorFlow\n- Web 后端：Django、Flask、FastAPI\n- 爬虫：requests、scrapy、playwright\n- 自动化：selenium、paramiko、openpyxl\n- 科学计算、图像处理、命令行工具等\n\n很多领域不需要从零造轮子。\n\n### 4. 是数据、AI 和科研领域的事实标准之一\n如果涉及数据分析、机器学习、人工智能、科学计算，Python 几乎是默认语言。即使主语言是 Java/C++/Go，也常用 Python 做数据分析、模型实验和工具链。\n\n### 5. 很适合做“胶水语言”\nPython 容易和其他语言、系统、服务集成：\n- 调用 C/C++ 扩展\n- 包装命令行工具\n- 连接数据库、消息队列、HTTP 服务\n- 在复杂系统中做集成脚本和运维工具\n\n所以它能补足主语言不擅长的场景。\n\n### 6. 后端和快速开发效率高\n用 FastAPI、Flask、Django 可以很快搭建 API、管理后台、内部系统。对创业项目、内部工具、原型验证特别友好。\n\n### 7. 测试和运维友好\nPython 常用于写自动化测试、CI/CD 脚本、监控脚本、云资源管理工具。很多 DevOps 工具也支持 Python SDK。\n\n### 8.

In [21]:
import os
import httpx
from unittest.mock import patch
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek
from openai import RateLimitError

load_dotenv()

request = httpx.Request("GET", "/")
response = httpx.Response(200, request=request)
error = RateLimitError("rate limit", response=response, body="")

qw_llm = ChatOpenAI(
    model="qwen3.8-max",
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url"),
    temperature=0,
    max_retries=0,
)

ds_llm = ChatDeepSeek(
    model="deepseek-ai/DeepSeek-V4.1-Flash",
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url"),
    temperature=0,
    max_retries=0,
)

llm = qw_llm.with_fallbacks([ds_llm], exceptions_to_handle=(RateLimitError,))

# 👇 关键：patch.object 打实例，不影响 ds_llm
with patch.object(qw_llm.client, "create", side_effect=error):
    try:
        result = llm.invoke("为什么程序员要学会Python?")
        print("✅ 回退成功，回答来自备用模型：")
        print(result.content)
    except RateLimitError:
        print("❌ Hit error（备用模型也失败了）")

✅ 回退成功，回答来自备用模型：
程序员不一定要学 Python，但它确实是一门“性价比很高”的通用语言。原因主要有这些：

1. **语法简单，上手快**
   Python 接近自然语言，代码可读性强。程序员可以把精力放在解决问题、理解业务和设计架构上，而不是被复杂语法绊住。

2. **生态极其丰富**
   无论做 Web、自动化、数据处理、爬虫、AI、测试、运维、桌面工具，几乎都有成熟库。例如：
   - Web：Django、Flask、FastAPI
   - 数据：pandas、NumPy
   - AI/机器学习：PyTorch、TensorFlow、scikit-learn
   - 自动化：requests、Selenium、Playwright
   - 运维：Ansible、Fabric
   - 测试：pytest

3. **最适合写“胶水代码”和自动化脚本**
   程序员日常有很多重复工作：批量改文件、处理日志、调用 API、生成报表、部署脚本、数据清洗。Python 写这些小工具非常快，能显著提升效率。

4. **数据与 AI 时代的通用语言**
   现在数据分析、机器学习、大模型应用、数据工程中，Python 几乎是事实标准。即使你不做算法，理解 Python 也更容易和算法、数据团队协作。

5. **原型开发快**
   想验证一个想法、做一个 MVP、写一个内部工具，Python 开发速度很快。适合先跑通，再决定是否用其他语言重写。

6. **跨领域能力强**
   Python 不局限于某一方向。一个后端程序员可以用它写脚本，一个前端可以用它做自动化，一个测试可以用它做接口测试，一个运维可以用它做平台工具。

7. **社区大、资料多**
   遇到问题容易搜到答案，第三方库多，招聘需求也广。对初学者和转方向的人都比较友好。

8. **有助于理解编程思想**
   Python 支持面向对象、函数式、过程式等多种风格，也适合学习数据结构、算法、网络、并发等基础概念。

9. **就业面广**
   Python 在后端、数据分析、人工智能、自动化测试、运维开发、爬虫、量化等领域都有岗位。虽然很多岗位还要求其他技能，但 Python 常是加分项甚至必备项。

10. **提升程序员的生产力**
   最重要的不是“